# ModelGuard — AI Project Readiness Reviewer

ModelGuard reviews an AI, Machine Learning, or Data Science project from a public GitHub repository.

It reads useful project files such as the README, Jupyter notebooks, Python files, and configuration files. It then extracts important project information, identifies missing details and risks, and generates an AI Project Readiness Report.

This tool does not provide legal, regulatory, ethical, or safety certification. It produces an educational review based only on the repository files that it can read.


## Business Challenge

AI projects often explain what a model does, but important information may still be missing.

For example:

- The intended users may not be clearly defined
- The dataset may not be properly described
- Important evaluation metrics may be missing
- Bias, privacy, security, and monitoring may not be discussed
- The project may not explain its limitations
- The deployment plan may not be clear

ModelGuard helps organize the available information and highlights areas that may need improvement.


## How ModelGuard Works

ModelGuard follows four main steps:

1. Read useful files from a public GitHub repository
2. Extract important project facts
3. Identify risks, gaps, and unanswered questions
4. Generate a final AI Project Readiness Report


In [ ]:
# imports

import os
import json
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI


## Initialize OpenAI

The OpenAI API key is loaded from the `.env` file.

The same OpenAI client will be reused for all LLM calls in ModelGuard.


In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key) > 10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key")

MODEL = "gpt-5-nano"
REPORT_MODEL = "gpt-4.1-mini"

openai = OpenAI()


# First Step: Read the Project from GitHub

ModelGuard accepts the URL of a public GitHub repository.

It checks the repository structure and reads useful text-based files such as:

- `README.md`
- Jupyter notebooks (`.ipynb`)
- Python files (`.py`)
- Dependency files
- Configuration files
- Model cards and dataset cards

Large files, images, datasets, model weights, and generated folders are ignored.


In [ ]:
# Public GitHub repository to review

github_url = "https://github.com/damaniayesh/Analytics-in-Sales"


## Extract the Repository Details

A GitHub repository URL normally follows this format:

`https://github.com/owner/repository`

The following function extracts the owner and repository name from the URL.


In [ ]:
# Extract the repository owner and repository name

def get_github_repository_details(github_url):
    clean_url = github_url.rstrip("/")
    parts = clean_url.split("/")

    owner = parts[-2]
    repository = parts[-1]

    return owner, repository


In [ ]:
# Test the repository details

repository_owner, repository_name = get_github_repository_details(github_url)

print("Repository owner:", repository_owner)
print("Repository name:", repository_name)


## Find Useful Repository Files

The GitHub API provides the repository's default branch and file structure.

The following function keeps useful project files and ignores folders such as `.git`, `node_modules`, checkpoints, datasets, images, and model weights.


In [ ]:
# File types and folders used by ModelGuard

USEFUL_EXTENSIONS = (
    ".md",
    ".py",
    ".ipynb",
    ".txt",
    ".json",
    ".yml",
    ".yaml",
    ".toml"
)

USEFUL_FILENAMES = {
    "requirements.txt",
    "pyproject.toml",
    "environment.yml",
    "environment.yaml",
    "Pipfile",
    "Dockerfile"
}

IGNORED_PARTS = {
    ".git",
    ".github",
    ".ipynb_checkpoints",
    "__pycache__",
    "node_modules",
    "venv",
    ".venv",
    "data",
    "datasets",
    "images",
    "assets",
    "checkpoints",
    "models",
    "weights"
}


In [ ]:
# Get the default branch and repository file tree

def get_repository_files(github_url):
    owner, repository = get_github_repository_details(github_url)

    repository_url = f"https://api.github.com/repos/{owner}/{repository}"
    repository_response = requests.get(repository_url, timeout=20)
    repository_response.raise_for_status()

    default_branch = repository_response.json()["default_branch"]

    tree_url = (
        f"https://api.github.com/repos/{owner}/{repository}/git/trees/"
        f"{default_branch}?recursive=1"
    )
    tree_response = requests.get(tree_url, timeout=20)
    tree_response.raise_for_status()

    files = []

    for item in tree_response.json().get("tree", []):
        if item.get("type") != "blob":
            continue

        path = item["path"]
        path_parts = set(path.split("/"))
        filename = path.split("/")[-1]

        if path_parts.intersection(IGNORED_PARTS):
            continue

        if filename in USEFUL_FILENAMES or path.lower().endswith(USEFUL_EXTENSIONS):
            files.append(path)

    return default_branch, files


In [ ]:
# View the useful files found in the repository

default_branch, repository_files = get_repository_files(github_url)

print("Default branch:", default_branch)
print("Useful files found:", len(repository_files))

repository_files[:30]


## Read the Repository Files

Text files can be read directly.

A Jupyter notebook is stored as JSON, so the following function extracts only its Markdown and code cells. Notebook outputs are ignored.


In [ ]:
# Convert a Jupyter notebook into readable text

def extract_notebook_text(notebook_content):
    notebook = json.loads(notebook_content)
    notebook_text = []

    for cell in notebook.get("cells", []):
        cell_type = cell.get("cell_type")
        source = "".join(cell.get("source", []))

        if not source.strip():
            continue

        if cell_type == "markdown":
            notebook_text.append(f"### Markdown Cell\n{source}")
        elif cell_type == "code":
            notebook_text.append(f"### Code Cell\n{source}")

    return "\n\n".join(notebook_text)


In [ ]:
# Download and read one repository file

def fetch_repository_file(owner, repository, branch, path):
    raw_url = (
        f"https://raw.githubusercontent.com/"
        f"{owner}/{repository}/{branch}/{path}"
    )

    response = requests.get(raw_url, timeout=20)
    response.raise_for_status()

    content = response.text

    if path.lower().endswith(".ipynb"):
        return extract_notebook_text(content)

    return content


## Combine the Project Information

ModelGuard reads a limited number of useful files and combines them into one text input.

Each file is labeled with its path so the LLM can understand where the information came from. Character limits prevent very large repositories from creating an oversized prompt.


In [ ]:
# Collect useful project information from the repository

def collect_project_information(
    github_url,
    max_files=15,
    max_chars_per_file=8_000,
    max_total_chars=50_000
):
    owner, repository = get_github_repository_details(github_url)
    branch, files = get_repository_files(github_url)

    priority_files = sorted(
        files,
        key=lambda path: (
            0 if path.lower().endswith("readme.md") else
            1 if path.lower().endswith(".ipynb") else
            2 if path.lower().endswith(".py") else
            3
        )
    )

    project_parts = []
    selected_files = []
    total_chars = 0

    for path in priority_files[:max_files]:
        try:
            content = fetch_repository_file(
                owner,
                repository,
                branch,
                path
            )
        except Exception as error:
            print(f"Skipped {path}: {error}")
            continue

        content = content[:max_chars_per_file]
        section = f"\n\n## File: {path}\n\n{content}"

        if total_chars + len(section) > max_total_chars:
            break

        project_parts.append(section)
        selected_files.append(path)
        total_chars += len(section)

    return repository, selected_files, "".join(project_parts)


In [ ]:
# Collect the project information

project_name, selected_files, project_information = collect_project_information(
    github_url
)

print("Project name:", project_name)
print("Files selected:", len(selected_files))
print("Characters collected:", len(project_information))

selected_files


In [ ]:
# Preview the collected project information

print(project_information[:5000])


# Second Step: Extract the Project Facts

The first LLM call reads the selected GitHub repository files and converts the available information into structured JSON.

It extracts details such as:

- Project purpose
- Intended users
- Model or system
- Dataset information
- Evaluation metrics
- Deployment plan
- Known limitations
- Monitoring plan

The model should not invent missing information.


In [ ]:
# Define the project extraction system prompt

extraction_system_prompt = """
You analyze information about a Data Science, Machine Learning, or Generative AI project.

Extract the important project facts from the supplied GitHub repository files.

Respond in valid JSON using exactly this structure:

{
    "project_name": "",
    "project_purpose": "",
    "intended_users": [],
    "model_or_system": "",
    "dataset_information": "",
    "input_features_or_sources": [],
    "evaluation_metrics": [],
    "deployment_plan": "",
    "known_limitations": [],
    "monitoring_plan": "",
    "missing_information": []
}

Rules:

- Use only information present in the supplied repository files.
- Do not guess or invent project details.
- Use "Not provided" when a detail is missing.
- Add important missing details to missing_information.
- Return JSON only.
"""


In [ ]:
# Define the project extraction user prompt

def get_extraction_user_prompt(project_name, project_information):
    user_prompt = f"""
Here are selected files from a GitHub project called {project_name}. 

Extract and organize the important project facts.

GitHub repository files:

{project_information}
"""
    return user_prompt


In [ ]:
# Preview the extraction prompt

print(get_extraction_user_prompt(project_name, project_information))


## Extract the Project Information

The following function sends the collected repository information to the model.

The response is requested in JSON so that the extracted information can be reused in later steps.


In [ ]:
# Extract the project facts

def extract_project_facts(project_name, project_information):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": extraction_system_prompt},
            {"role": "user", "content": get_extraction_user_prompt(project_name,project_information)
            }
        ],
        response_format={"type": "json_object"}
    )

    result = response.choices[0].message.content
    return json.loads(result)


In [ ]:
# Run the first LLM call

project_facts = extract_project_facts(project_name,project_information)

project_facts


# Third Step: Identify Risks and Missing Information

The second LLM call reviews the extracted project facts.

It looks for possible gaps related to:

- Data quality
- Model evaluation
- Bias and fairness
- Privacy and security
- Deployment
- Monitoring
- Documentation

A missing detail should be reported as a gap, not as proof that the project is unsafe.


In [ ]:
# Define the risk-review system prompt

risk_system_prompt = """
You are an AI project reviewer. Review the supplied project facts and identify strengths, risks, documentation gaps, and unanswered questions.

Respond in valid JSON using exactly this structure:

{
    "strengths": [],
    "data_risks": [],
    "evaluation_gaps": [],
    "bias_and_fairness_gaps": [],
    "privacy_and_security_gaps": [],
    "deployment_risks": [],
    "monitoring_gaps": [],
    "recommended_actions": [],
    "readiness_status": ""
}

The readiness_status must be exactly one of:

- "Ready for limited testing"
- "Needs improvement"
- "Not ready"

Rules:

- Base the review only on the supplied project facts.
- Do not invent confirmed risks.
- Treat missing information as a gap or unanswered question.
- Do not claim legal, regulatory, ethical, or safety certification.
- Recommend practical next actions.
- Return JSON only.
"""


In [ ]:
# Define the risk-review user prompt

def get_risk_user_prompt(project_facts):
    user_prompt = """
Review the following extracted AI project facts. Identify the strengths, risks, missing evidence, and recommended actions.

Project facts:

"""
    user_prompt += json.dumps(project_facts, indent=2)
    return user_prompt


In [ ]:
# Preview the risk-review prompt

print(get_risk_user_prompt(project_facts))


## Analyze the Project Risks

The following function sends the structured project facts to the model.

The model returns a structured risk and readiness review.


In [ ]:
# Analyze project risks and gaps

def analyze_project_risks(project_facts):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": risk_system_prompt},
            {"role": "user","content": get_risk_user_prompt(project_facts)
}
        ],
        response_format={"type": "json_object"}
    )

    result = response.choices[0].message.content
    return json.loads(result)


In [ ]:
# Run the second LLM call

risk_analysis = analyze_project_risks(project_facts)

risk_analysis


# Fourth Step: Create the Readiness Report

The third LLM call combines:

- The extracted project facts
- The risk and gap analysis
- The recommended actions
- The readiness status

It creates a final AI Project Readiness Report in Markdown.


In [ ]:
# Define the final report system prompt

report_system_prompt = """
You create clear and practical AI Project Readiness Reports.

Use the supplied project facts and risk analysis to produce a structured report.

Use exactly these markdown sections:

# AI Project Readiness Report

## Project Overview

## Intended Use and Users

## Model and Data Summary

## Reported Evaluation

## Strengths

## Missing Information

## Risk and Evaluation Review

## Recommended Actions

## Readiness Status

## Important Note

Rules:

- Use only the supplied evidence.
- Do not invent project details.
- Clearly separate reported facts from missing information.
- Do not claim legal, regulatory, ethical, or safety certification.
- Keep the report useful for technical and non-technical readers.
- Preserve the readiness status from the risk analysis.
- State that the report is an educational review based only on the supplied information.
- Respond in markdown without code blocks.
"""


In [ ]:
# Define the final report user prompt

def get_report_user_prompt(project_facts, risk_analysis):
    user_prompt = """
Create an AI Project Readiness Report using the information below.

Project facts:

"""
    user_prompt += json.dumps(project_facts, indent=2)

    user_prompt += """

Risk analysis:

"""
    user_prompt += json.dumps(risk_analysis, indent=2)

    return user_prompt


In [ ]:
# Preview the final report prompt

print(get_report_user_prompt(project_facts, risk_analysis))


In [ ]:
# Create the final AI Project Readiness Report

def create_readiness_report(project_facts, risk_analysis):
    response = openai.chat.completions.create(
        model=REPORT_MODEL,
        messages=[
            {"role": "system", "content": report_system_prompt},
            {
                "role": "user",
                "content": get_report_user_prompt(
                    project_facts,
                    risk_analysis
                )
            }
        ]
    )

    result = response.choices[0].message.content
    display(Markdown(result))

    return result


In [ ]:
# Run the third LLM call

readiness_report = create_readiness_report(
    project_facts,
    risk_analysis
)


In [ ]:
# Stream the final AI Project Readiness Report

def stream_readiness_report(project_facts, risk_analysis):
    stream = openai.chat.completions.create(
        model=REPORT_MODEL,
        messages=[
            {"role": "system", "content": report_system_prompt},
            {"role": "user","content": get_report_user_prompt(project_facts,risk_analysis)
            }
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(
            Markdown(response),
            display_id=display_handle.display_id
        )

    return response


In [ ]:
# Generate the streamed report

streamed_readiness_report = stream_readiness_report(project_facts,risk_analysis)


# Limitations

ModelGuard has several important limitations:

- It reviews only selected readable repository files
- It does not run or test the project code
- It does not inspect datasets, images, model weights, or binary files
- It cannot verify whether reported metrics are correct
- Large repositories may be partially analyzed because of file and character limits
- Private repositories require authenticated GitHub access, which is not included in this version
- It does not provide legal or regulatory certification


# Future Improvements

Possible future improvements include:

1. Letting an LLM choose the most relevant files from the repository tree
2. Supporting private repositories with a GitHub token
3. Inspecting model cards and dataset cards more deeply
4. Running static code checks and dependency analysis
5. Linking each finding to the exact source file
6. Adding risk severity levels
7. Exporting the final report as a PDF
8. Adding a Gradio user interface


# Conclusion

ModelGuard demonstrates a multi-step LLM workflow.

It collects useful information from a public GitHub repository, including README files, Jupyter notebooks, Python files, and configuration files.

The first LLM call converts the repository information into structured project facts.

The second LLM call identifies risks, gaps, and recommended actions.

The third LLM call combines the results into an AI Project Readiness Report.


# Week 2 Upgrade - Gradio User Interface

Now that ModelGuard can review a GitHub repository and create an AI Project Readiness Report, I can make it easier to use.

In this upgrade I will:

1. Add a Gradio user interface
2. Let the user enter a GitHub repository URL
3. Run the complete ModelGuard workflow
4. Stream the final report
5. Add simple authentication

In [ ]:
# importing Libraries
import gradio as gr

## Create the ModelGuard Gradio Function

The user will enter the URL of a public GitHub repository.

ModelGuard will then:

- Read the useful repository files
- Extract the project facts
- Analyze risks and missing information
- Generate the final AI Project Readiness Report

In [ ]:
# Run the complete ModelGuard workflow

def review_project(github_url):
    
    yield "Reading the GitHub repository..."
    
    project_name, selected_files, project_information = collect_project_information(
        github_url
    )
    
    yield "Extracting project information..."
    
    project_facts = extract_project_facts(
        project_name,
        project_information
    )
    
    yield "Analyzing risks and missing information..."
    
    risk_analysis = analyze_project_risks(
        project_facts
    )
    
    yield "Creating the AI Project Readiness Report..."
    
    stream = openai.chat.completions.create(
        model=REPORT_MODEL,
        messages=[
            {"role": "system", "content": report_system_prompt},
            {
                "role": "user",
                "content": get_report_user_prompt(
                    project_facts,
                    risk_analysis
                )
            }
        ],
        stream=True
    )
    
    response = ""
    
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

## Build the Gradio Interface

Gradio lets us turn ModelGuard into a simple web application.

The user only needs to enter a public GitHub repository URL.

In [ ]:
# Gradio User Interface

github_input = gr.Textbox(label="GitHub Repository URL:",info="Enter the URL of a public GitHub repository")

report_output = gr.Markdown(label="AI Project Readiness Report:")

view = gr.Interface(
    fn=review_project,
    title="ModelGuard",
    description="Review an AI, Machine Learning, or Data Science project from a public GitHub repository.",
    inputs=[github_input],
    outputs=[report_output],
    examples=[
        ["https://github.com/damaniayesh/Analytics-in-Sales"]
    ],
    flagging_mode="never"
)

In [ ]:
view = gr.Interface(
    fn=review_project,
    title="ModelGuard",
    inputs=[github_input],
    outputs=[report_output],
    examples=[
        ["https://github.com/damaniayesh/Analytics-in-Sales"]
    ],
    flagging_mode="never"
)

In [ ]:
view.launch(
    inbrowser=True, share=True,
    auth=("yesh", "yesh@123")
)

# Conclusion

In this project, ModelGuard was upgraded with a Gradio user interface and simple authentication.

The user can now enter a public GitHub repository URL and generate an AI Project Readiness Report through a simple web interface.

This upgrade makes ModelGuard easier to use while keeping the original project analysis workflow unchanged.